In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze-ingest")
    .config("spark.jars", "/app/mariadb-connector.jar")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

NOTE: Picked up JDK_JAVA_OPTIONS: --add-modules=jdk.incubator.vector
NOTE: Picked up JDK_JAVA_OPTIONS: --add-modules=jdk.incubator.vector
26/05/31 12:30:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
JDBC_URL = "jdbc:mysql://mariadb:3306/mydb?permitMysqlScheme"
JDBC_OPTS = {
    "user": "spark",
    "password": "sparkpassword",
    "driver": "org.mariadb.jdbc.Driver",
}

def export_table(table_name: str):
    df = (
        spark.read.format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", table_name)
        .options(**JDBC_OPTS)
        .load()
    )

    df.write.mode("overwrite").parquet(f"/app/data/bronze/{table_name}")

tables = ["tabProyecto", "tabCliente"]

for table_name in tables:
    export_table(table_name)